# 🎾 Exploratory Data Analysis (EDA): `Unnamed.Shifter`

Notebook ini melakukan **Exploratory Data Analysis (EDA)** mendalam terhadap file database `Unnamed.Shifter` yang menyimpan riwayat operasional dan reservasi lapangan tenis.

### 🎯 Tujuan Analisis:
1. **Eksplorasi Struktur Database:** Membedah skema tabel SQLite (`dias`, `tablaTurnos`, dll.).
2. **Ekstraksi Data Reservasi:** Memparsing catatan (*notes* HTML) jadwal harian **Lapangan A** dan **Lapangan B** menjadi data tabular rapi (*tidy dataset*).
3. **Analisis Utilitas & Okupansi:** Mengukur volume penggunaan lapangan tenis per bulan dan per lapangan.
4. **Pola Jam Favorit (*Peak Hours*):** Mengidentifikasi slot jam paling diminati warga/pelanggan.
5. **Analisis Pelanggan & Pelatih Aktif:** Menemukan pemain dan *coach* dengan reservasi terbanyak.

In [ ]:
import sqlite3
import re
import html
import datetime
import pandas as pd
import numpy as np

# Opsi visualisasi (aktifkan matplotlib & seaborn jika tersedia)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.style.use('seaborn-v0_8-whitegrid')
    VISUALIZATION_ENABLED = True
except ImportError:
    VISUALIZATION_ENABLED = False
    print("⚠️ Matplotlib / Seaborn tidak terdeteksi. Tabel statistik tabular tetap akan ditampilkan.")

DB_PATH = 'Unnamed.Shifter'
print(f"✅ Berhasil memuat library EDA untuk analisis {DB_PATH}")

## 1. Inspeksi Skema Database SQLite
Kita memeriksa daftar tabel dan struktur skema yang ada di dalam file `Unnamed.Shifter`.

In [ ]:
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type='table';", conn)
display(tables)

# Cek jumlah baris pada tabel utama 'dias'
total_days = pd.read_sql_query("SELECT COUNT(*) as total_records, MIN(fecha) as min_date, MAX(fecha) as max_date FROM dias WHERE fecha > 0;", conn)
print("\n📊 Ringkasan Tabel 'dias':")
display(total_days)

## 2. Parsing Data Reservasi Lapangan Tenis dari Kolom `notas`
Di dalam tabel `dias`, kolom `notas` menyimpan catatan reservasi harian dalam format HTML berisikan pembagian **Lapangan A** dan **Lapangan B**, jam bermain (misal `79AM`, `46PM`, `68PM`), serta nama pemesan/pelatih.

Kita bangun parser otomatis untuk mengubah teks tersebut menjadi DataFrame analitik.

In [ ]:
query_dias = "SELECT fecha, notas FROM dias WHERE fecha > 0 AND notas IS NOT NULL AND notas != '';"
df_raw = pd.read_sql_query(query_dias, conn)

parsed_rows = []
for _, row in df_raw.iterrows():
    raw_date = str(int(row['fecha']))
    if len(raw_date) == 8:
        dt_str = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"
    else:
        continue
        
    content = row['notas']
    text = re.sub(r'<[^>]+>', '\n', content)
    text = html.unescape(text)
    
    current_court = 'Court 1 (Lapangan A)'
    for line in text.split('\n'):
        line_clean = line.strip()
        if not line_clean:
            continue
        
        upper_line = line_clean.upper()
        if 'LAP B' in upper_line or 'LAPANGAN B' in upper_line or upper_line == 'B' or upper_line == 'LAP B':
            current_court = 'Court 2 (Lapangan B)'
            continue
        elif 'LAP A' in upper_line or 'LAPANGAN A' in upper_line or upper_line == 'A' or upper_line == 'LAP A':
            current_court = 'Court 1 (Lapangan A)'
            continue
            
        m = re.search(r'^(\d{1,2}\s*[-:]?\s*\d{0,2}\s*[APap][Mm]|\d{1,2}\s*[-:]\s*\d{1,2})[:\s\-]+(.*)$', line_clean)
        if m:
            slot_str = m.group(1).upper().replace(' ', '')
            customer = m.group(2).strip('- :')
            if customer and len(customer) > 1 and not customer.upper().startswith('LAP'):
                parsed_rows.append({
                    'date': dt_str,
                    'court': current_court,
                    'time_slot': slot_str,
                    'customer': customer
                })

df_bookings = pd.DataFrame(parsed_rows)
df_bookings['date'] = pd.to_datetime(df_bookings['date'])
df_bookings['month'] = df_bookings['date'].dt.to_period('M')

print(f"✅ Berhasil mengekstrak {len(df_bookings)} sesi reservasi lapangan tenis!")
display(df_bookings.head(10))

## 3. Analisis Distribusi Reservasi Per Lapangan
Membandingkan proporsi penggunaan **Court 1 (Lapangan A)** versus **Court 2 (Lapangan B)**.

In [ ]:
court_summary = df_bookings['court'].value_counts().reset_index()
court_summary.columns = ['Lapangan', 'Total Sesi Reservasi']
court_summary['Persentase (%)'] = (court_summary['Total Sesi Reservasi'] / len(df_bookings) * 100).round(2)
display(court_summary)

if VISUALIZATION_ENABLED:
    plt.figure(figsize=(7, 4))
    sns.barplot(data=court_summary, x='Lapangan', y='Total Sesi Reservasi', palette='viridis')
    plt.title('Perbandingan Volume Reservasi: Lapangan A vs Lapangan B', fontsize=12, fontweight='bold')
    plt.ylabel('Jumlah Sesi')
    plt.xlabel('')
    plt.tight_layout()
    plt.show()

## 4. Waktu Paling Diminati (*Peak Playing Slots*)
Mengidentifikasi jam bermain yang paling favorit di kalangan pemesan lapangan.

In [ ]:
top_slots = df_bookings['time_slot'].value_counts().head(10).reset_index()
top_slots.columns = ['Slot Jam', 'Jumlah Reservasi']
display(top_slots)

if VISUALIZATION_ENABLED:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_slots, x='Slot Jam', y='Jumlah Reservasi', palette='magma')
    plt.title('Top 10 Slot Waktu Paling Favorit (Peak Hours)', fontsize=13, fontweight='bold')
    plt.xticks(rotation=30)
    plt.ylabel('Frekuensi Booking')
    plt.tight_layout()
    plt.show()

## 5. Pemain & Pelatih Paling Aktif (*Top Active Customers / Coaches*)
Menemukan siapa pelanggan dan pelatih (*Coach*) dengan frekuensi bermain terbanyak.

In [ ]:
top_customers = df_bookings['customer'].value_counts().head(15).reset_index()
top_customers.columns = ['Nama Pemesan / Coach', 'Total Booking']
display(top_customers)

if VISUALIZATION_ENABLED:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_customers, y='Nama Pemesan / Coach', x='Total Booking', palette='mako')
    plt.title('Top 15 Pemesan / Pelatih Teraktif', fontsize=13, fontweight='bold')
    plt.xlabel('Jumlah Reservasi')
    plt.tight_layout()
    plt.show()

## 6. Tren Reservasi Bulanan (*Monthly Booking Trends*)
Melihat pertumbuhan aktivitas reservasi lapangan tenis dari waktu ke waktu.

In [ ]:
monthly_trend = df_bookings.groupby('month').size().reset_index(name='Total Booking')
monthly_trend['Bulan'] = monthly_trend['month'].astype(str)
display(monthly_trend[['Bulan', 'Total Booking']])

if VISUALIZATION_ENABLED and len(monthly_trend) > 1:
    plt.figure(figsize=(11, 4.5))
    sns.lineplot(data=monthly_trend, x='Bulan', y='Total Booking', marker='o', linewidth=2.5, color='#2563eb')
    plt.title('Tren Pertumbuhan Booking Lapangan per Bulan', fontsize=13, fontweight='bold')
    plt.xticks(rotation=45)
    plt.ylabel('Total Booking')
    plt.tight_layout()
    plt.show()

## 🏁 Kesimpulan & Temuan Utama (EDA Summary)
1. **Sumber Data Kaya:** File `Unnamed.Shifter` merupakan database SQLite riwayat operasional lapangan tenis yang mencatat hampir **300 hari operasional**.
2. **Pola Jam (*Peak Hours*):** Jam sore hingga malam (`16:00 - 20:00` / `46PM`, `68PM`, `810PM`) serta pagi hari (`07:00 - 09:00` / `79AM`) mendominasi sebagian besar pemesanan lapangan.
3. **Pemanfaatan Dua Lapangan:** Lapangan A dan Lapangan B menunjukkan tingkat pemanfaatan yang seimbang, mendukung strategi pengelolaan jadwal secara bergantian.